# 15 · Testing and Debugging

After this notebook you can write unit tests in both `unittest` and pytest style, mock an LLM client so code that calls a model is tested deterministically and for free, apply the testing techniques that work for LLM apps (fakes, property checks, golden files), and find bugs and slow spots with tracebacks, logging and a profiler.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [12 · Decorators and context managers](12_decorators_and_context_managers.ipynb) (`patch` is both), [13 · Type hints and Pydantic](13_type_hints_and_pydantic.ipynb)

## Why this matters in interviews

- "How do you test an LLM feature in CI without spending a fortune?" and "How do you test an agent?" are standard questions. The answer is mostly *ordinary* testing of the code around the model — with the model mocked.
- Live-coding rounds increasingly say "now write a test for it". Knowing pytest idioms (plain `assert`, fixtures, `parametrize`) and `unittest.mock` is expected.
- Debugging and profiling questions ("this endpoint got slow — what do you do?") reward a method: reproduce, read the traceback, measure before optimising.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pd29` | How do you test an LLM feature in CI without spending a fortune? |
| `ag28` | How do you test an agent in CI? |
| `in32` | How do you make LLM output reproducible for testing? |
| `ev22` | What is regression testing for an LLM system, and how is it different? |

## 1. `assert` — the smallest test

`assert condition, "message"` raises `AssertionError` with the message if the condition is false. It is perfect for tests and for checking your own assumptions — but **not** for validating user input: `python -O` strips every `assert` out of the program.

In [ ]:
import difflib
import doctest
import importlib.util
import io
import json
import os
import random
import re
import sys
import time
import unittest
from unittest import mock

LABELS = ("billing", "shipping", "technical", "other")

def parse_label(reply):
    """Turn a model's free-text reply into one of LABELS (fallback: 'other')."""
    word = re.sub(r"[^a-z]", "", (reply or "").strip().split()[0].lower()) if (reply or "").strip() else ""
    return word if word in LABELS else "other"

assert parse_label("Billing") == "billing"
assert parse_label("  shipping.  ") == "shipping"
assert parse_label("I think it's about a refund") == "other", "unknown words must fall back to 'other'"
try:
    assert parse_label("") == "billing", "empty reply should never be billing"
except AssertionError as e:
    print("AssertionError:", e)

## 2. Writing testable code

Code is easy to test when the logic is **pure** (inputs in, value out) and anything slow, random or external is **passed in** rather than reached for. For an LLM feature that means: build the prompt in one function, parse the reply in another, and pass the client in as a parameter (dependency injection), so a test can hand it a fake.

| Hard to test | Easy to test |
|---|---|
| creates `OpenAI()` inside the function | takes `client` as an argument |
| calls `time.sleep`, `random`, `datetime.now()` directly | takes `sleep`, `rng`, `now` (or patch them) |
| prints results | returns results |
| one 80-line function: prompt + call + parse + save | small functions you can test one by one |

In [ ]:
SYSTEM = f"Classify the support ticket. Reply with exactly one word: {', '.join(LABELS)}."

def build_messages(ticket):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ticket.strip()}]

def classify_ticket(ticket, client, model="gpt-4.1-mini"):
    """Prompt -> model -> parsed label. The client is injected, so tests can pass a fake."""
    resp = client.chat.completions.create(model=model, messages=build_messages(ticket), temperature=0)
    return parse_label(resp.choices[0].message.content)

print(build_messages("  My card was charged twice  "))

## 3. `unittest` inside a notebook

`unittest` ships with Python: subclass `TestCase`, write `test_*` methods, use `self.assertEqual`, `assertIn`, `assertRaises`… `subTest` reports each case of a loop separately. In a script you run `python -m unittest`; in a notebook, `unittest.main(argv=["ignored"], exit=False)` — the dummy `argv` stops it reading the kernel's own command-line arguments, and `exit=False` stops it calling `sys.exit`. (It runs every `TestCase` defined in the notebook so far.)

In [ ]:
class TestParseLabel(unittest.TestCase):
    def test_exact_labels(self):
        for label in LABELS:
            with self.subTest(label=label):
                self.assertEqual(parse_label(label), label)

    def test_case_and_punctuation(self):
        self.assertEqual(parse_label("  Technical!\n"), "technical")

    def test_unknown_and_empty_fall_back(self):
        for reply in ["refund please", "", None, "   "]:
            with self.subTest(reply=reply):
                self.assertEqual(parse_label(reply), "other")

    def test_messages_have_system_first(self):
        roles = [m["role"] for m in build_messages("hi")]
        self.assertEqual(roles, ["system", "user"])

program = unittest.main(argv=["ignored"], exit=False, testRunner=unittest.TextTestRunner(stream=sys.stdout, verbosity=2))
assert program.result.wasSuccessful()

What does a **failure** look like? Here is a deliberately wrong expectation, run on its own with a loader, so you can see the report you will be reading in CI:

In [ ]:
class TestDeliberatelyWrong(unittest.TestCase):
    def test_wrong_expectation(self):
        self.assertEqual(build_messages("hi")[0]["role"], "user", "system prompt must come first")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestDeliberatelyWrong)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(suite)
assert len(result.failures) == 1
del TestDeliberatelyWrong        # so a later unittest.main() does not pick it up

## 4. pytest style — and a tiny runner to see how it works

Most Python teams use **pytest**. Its style is simpler than `unittest`: tests are plain functions using plain `assert` (pytest rewrites them to show the values on failure), shared setup comes from **fixtures** injected by parameter name, and **parametrize** turns one test into many cases.

In [ ]:
print("pytest installed here?", importlib.util.find_spec("pytest") is not None, "(we don't install it; the runner below mimics it)")

What you would write in `tests/test_classify.py`, run with `pytest -q` (`-k name` to filter, `-x` to stop at the first failure, `--lf` to rerun the last failures):

```python
import pytest

@pytest.fixture
def fake_client():                       # injected into any test that has a `fake_client` parameter
    return make_fake_client(reply="billing")

@pytest.mark.parametrize("reply, expected", [("Billing", "billing"), ("shipping.", "shipping"), ("??", "other")])
def test_parse_label(reply, expected):
    assert parse_label(reply) == expected

def test_classify_sends_temperature_zero(fake_client):
    classify_ticket("charged twice", fake_client)
    assert fake_client.chat.completions.create.call_args.kwargs["temperature"] == 0

def test_empty_ticket_raises():
    with pytest.raises(ValueError):
        validate_ticket("")
```

Built-in fixtures worth naming: `tmp_path` (a fresh temp folder), `monkeypatch` (set env vars and attributes, undone after the test), `capsys` (captured output). The toy runner below implements the two core ideas — fixture injection **by parameter name** and parametrize — in about 30 lines, so they stop being magic:

In [ ]:
import inspect

FIXTURES = {}

def fixture(fn):
    FIXTURES[fn.__name__] = fn
    return fn

def parametrize(names, cases):
    def decorator(fn):
        fn.cases = [dict(zip([n.strip() for n in names.split(",")], case)) for case in cases]
        return fn
    return decorator

def run_tests(namespace):
    passed = failed = 0
    for name, fn in sorted(namespace.items()):
        if not (name.startswith("test_") and callable(fn)) or isinstance(fn, type):
            continue
        for case in getattr(fn, "cases", [{}]):
            kwargs = dict(case)
            for param in inspect.signature(fn).parameters:        # inject fixtures by name, like pytest
                if param not in kwargs:
                    kwargs[param] = FIXTURES[param]()
            label = f"{name}[{', '.join(map(repr, case.values()))}]" if case else name
            try:
                fn(**kwargs)
                passed += 1
                print(f"PASSED {label}")
            except AssertionError as e:
                failed += 1
                print(f"FAILED {label}: {e}")
    print(f"== {passed} passed, {failed} failed ==")
    return failed == 0

In [ ]:
def make_fake_client(reply):
    """A MagicMock shaped like OpenAI's client (explained in the next section)."""
    fake = mock.MagicMock()
    fake.chat.completions.create.return_value.choices = [mock.Mock(message=mock.Mock(content=reply))]
    return fake

@fixture
def fake_client():
    return make_fake_client("Billing")

@parametrize("reply, expected", [("Billing", "billing"), ("shipping.", "shipping"), ("??", "other")])
def test_parse_label(reply, expected):
    assert parse_label(reply) == expected, f"{reply!r} -> {parse_label(reply)!r}"

def test_classify_uses_temperature_zero(fake_client):
    assert classify_ticket("charged twice", fake_client) == "billing"
    assert fake_client.chat.completions.create.call_args.kwargs["temperature"] == 0

assert run_tests(dict(globals()))

## 5. `unittest.mock`: `MagicMock`, `side_effect`, `patch`

A **mock** is an object that accepts any attribute access or call, records what happened, and returns what you configure. That is exactly what you need to stand in for an LLM client:

- `return_value` — what a call returns; `side_effect` — raise an exception, return a **sequence** of values one per call, or run a function;
- assertions on the record: `assert_called_once_with`, `call_count`, `call_args.kwargs`;
- `spec=` / `create_autospec` — make the mock reject attributes the real object does not have, so typos fail instead of silently passing.

For realism, the fake below returns a genuine `openai` `ChatCompletion` object, so the code under test parses exactly what the SDK would give it.

In [ ]:
from openai.types.chat import ChatCompletion

def completion(text):
    return ChatCompletion.model_validate({
        "id": "chatcmpl-test", "object": "chat.completion", "created": 0, "model": "gpt-4.1-mini",
        "choices": [{"index": 0, "finish_reason": "stop", "message": {"role": "assistant", "content": text}}]})

fake = mock.MagicMock()
fake.chat.completions.create.return_value = completion("Shipping")
label = classify_ticket("Where is my parcel? It has been 10 days.", fake)

call = fake.chat.completions.create.call_args
print("label:", label, "| calls:", fake.chat.completions.create.call_count)
print("sent model/temperature:", call.kwargs["model"], call.kwargs["temperature"])
print("sent user message:", call.kwargs["messages"][-1]["content"])
assert label == "shipping"
fake.chat.completions.create.assert_called_once()

sloppy = mock.MagicMock()
print("a plain MagicMock accepts typos:", type(sloppy.chat.completions.craete).__name__)
strict = mock.create_autospec(classify_ticket)          # autospec copies the real signature
try:
    strict("ticket", client=None, temprature=0)
except TypeError as e:
    print("autospec rejects a wrong argument:", e)

`side_effect` makes failure paths testable. Below, a small retry wrapper is tested against "timeout, then success" — and `patch("time.sleep")` stops the test from actually waiting.

**Patch where the name is looked up, not where it is defined.** The retry code calls `time.sleep(...)` through the module, so patching `time.sleep` works. If it had done `from time import sleep`, you would have to patch the name in *its* module (`__main__.sleep` here) — patching `time.sleep` would do nothing. This is the most common `mock.patch` mistake.

In [ ]:
def classify_with_retry(ticket, client, attempts=3):
    for attempt in range(attempts):
        try:
            return classify_ticket(ticket, client)
        except TimeoutError:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)                     # looked up on the time module at call time

fake = mock.MagicMock()
fake.chat.completions.create.side_effect = [TimeoutError("read timed out"), completion("technical")]
start = time.perf_counter()
with mock.patch("time.sleep") as fake_sleep:
    label = classify_with_retry("The app crashes on login", fake)
print(f"label={label} | calls={fake.chat.completions.create.call_count} | "
      f"sleep calls={fake_sleep.call_args_list} | test took {time.perf_counter() - start:.3f} s")
assert label == "technical" and fake_sleep.call_args_list == [mock.call(1)]

with mock.patch.dict(os.environ, {"LLM_MODEL": "test-model"}):   # env var only inside the block
    print("inside patch.dict:", os.environ["LLM_MODEL"])
print("after patch.dict :", os.environ.get("LLM_MODEL"))

## 6. Testing LLM applications

The model is non-deterministic, costs money and changes under you, so the testing strategy is layered (pd29, ag28):

| Layer | What it checks | Model | When |
|---|---|---|---|
| **Unit tests with fakes/mocks** | prompt building, parsing, validation, retries, control flow | mocked | every commit, seconds |
| **Record / replay (fixtures)** | real responses captured once, replayed forever | recorded | every commit |
| **Property / contract tests** | invariants: parses as JSON, label in the allowed set, citations point at provided docs, length limits | fake or real | every commit |
| **Golden files** | the exact rendered prompt / output format has not changed unexpectedly | none | every commit |
| **Small real smoke suite** | 10–20 real calls on a cheap model, with a cost ceiling | real | per merge |
| **Eval suite with a pass-rate gate** | quality, measured as "18/20 pass", never "all green" | real | nightly |

Assert on **structure and meaning**, not exact strings (in32): the parsed label, the JSON fields, the presence of a citation — an exact-string assertion on model output is a test that fails on a Tuesday for no reason. For an agent (ag28), assert on the **outcome and trajectory**: task done, no forbidden tool called, steps under a budget.

In [ ]:
class FakeLLM:
    """A deterministic stand-in: scripted replies, and a log of every prompt it was sent."""
    def __init__(self, replies):
        self.replies, self.prompts = list(replies), []

    def complete(self, prompt):
        self.prompts.append(prompt)
        return self.replies.pop(0)

def answer_with_citations(question, docs, llm):
    context = "\n".join(f"[{doc_id}] {text}" for doc_id, text in docs.items())
    reply = llm.complete(f"Answer from the context and cite sources like [id].\n{context}\nQ: {question}")
    return reply, re.findall(r"\[([\w.-]+)\]", reply)

docs = {"refunds.md": "Refunds are issued within 30 days.", "shipping.md": "Shipping takes 3-5 days."}
llm = FakeLLM(["Refunds take up to 30 days [refunds.md].", "Shipping is quick [pricing.md]."])

for question in ["How long do refunds take?", "How fast is shipping?"]:
    reply, cited = answer_with_citations(question, docs, llm)
    ok = bool(cited) and all(c in docs for c in cited)          # contract: every citation is a provided doc
    print(f"{'PASS' if ok else 'FAIL'}  cited={cited}  reply={reply!r}")
assert "refunds.md" in llm.prompts[0]                              # the context really reached the prompt

The second reply cites a document that was never provided — a hallucinated source. A property test catches it without knowing the "right" answer.

**Property-based testing** generates many random inputs and checks invariants that must always hold. (The `hypothesis` library does this properly, with shrinking to a minimal failing case; it is not installed here, so a seeded loop shows the idea.) Target: a chunker, where "every chunk respects the size limit" and "gluing chunks back together restores the text" must hold for *any* input.

In [ ]:
def chunk_text(text, size, overlap):
    """Split text into chunks of at most `size` characters, each overlapping the previous by `overlap`."""
    if not 0 <= overlap < size:
        raise ValueError("need 0 <= overlap < size")
    step = size - overlap
    return [text[i:i + size] for i in range(0, max(len(text) - overlap, 1), step)]

def reassemble(chunks, overlap):
    return chunks[0] + "".join(c[overlap:] for c in chunks[1:]) if chunks else ""

rng = random.Random(0)
cases = 0
for _ in range(500):
    text = "".join(rng.choice("abc de\n") for _ in range(rng.randint(1, 300)))
    size = rng.randint(2, 50)
    overlap = rng.randint(0, size - 1)
    chunks = chunk_text(text, size, overlap)
    assert all(len(c) <= size for c in chunks), (size, overlap)
    assert reassemble(chunks, overlap) == text, (text, size, overlap)
    cases += 1
print(f"{cases} random cases: size limit and lossless reassembly hold")

**Golden files** (ev22): save a known-good output once, then compare every future run against it and show a **diff** when it changes. Perfect for rendered prompts — a stray edit to a template changes every answer, and the diff makes it reviewable. When a change is intended, you regenerate the golden file deliberately (commonly with an `--update-golden` flag).

In [ ]:
import tempfile
from pathlib import Path

golden_dir = Path(tempfile.mkdtemp(prefix="nb15_golden_"))

def render_prompt(question, context, version=1):
    rules = "Answer only from the context." if version == 1 else "Answer from the context when possible."
    return f"You are Acme's support assistant.\n{rules}\nContext:\n{context}\nQuestion: {question}\n"

def check_golden(name, actual, update=False):
    path = golden_dir / f"{name}.txt"
    if update or not path.exists():
        path.write_text(actual, encoding="utf-8")
        return True, "golden file written"
    expected = path.read_text(encoding="utf-8")
    diff = "".join(difflib.unified_diff(expected.splitlines(True), actual.splitlines(True), "golden", "actual"))
    return actual == expected, diff

args = ("Can I return an e-book?", "Digital goods are non-refundable.")
print(check_golden("refund_prompt", render_prompt(*args)))                   # first run: records it
ok, diff = check_golden("refund_prompt", render_prompt(*args, version=2))  # someone "improved" the rules
print("matches golden?", ok)
print(diff)
assert check_golden("refund_prompt", render_prompt(*args))[0] and not ok

## 7. `doctest` — examples in the docstring that are also tests

`doctest` finds `>>>` examples in docstrings, runs them and compares the printed output. Good for small pure helpers: the documentation can never silently go out of date.

In [ ]:
def slugify(title):
    """Make a URL-safe slug.

    >>> slugify("Hello, RAG World!")
    'hello-rag-world'
    >>> slugify("  multiple   spaces  ")
    'multiple-spaces'
    """
    return re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")

def count_tokens(text):
    """Very rough token count.

    >>> count_tokens("one two three")
    4
    """
    return len(text.split())

buffer = io.StringIO()
doctest.run_docstring_examples(slugify, {"slugify": slugify}, verbose=False)   # silent = all passed
runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(count_tokens, globs={"count_tokens": count_tokens}):
    runner.run(test, out=buffer.write)
print("slugify examples: all passed (no output)")
print(re.sub(r'File "[^"]*[\\/]([^"\\/]+)"', r'File "\1"', buffer.getvalue().strip()))   # short file names
assert runner.failures == 1          # the count_tokens docstring example is deliberately wrong

## 8. Debugging: print vs logging, pdb, and tracebacks

- **`print` debugging** is fine for a minute; for anything you keep, use **logging** with levels. Leave `logger.debug(...)` lines in the code and turn them on with the level, no edits needed.
- **`breakpoint()`** (Python 3.7+) drops into **pdb** at that line. Core commands: `n` next line, `s` step into, `c` continue, `p expr` / `pp expr` print, `l` list source, `w` where (the stack), `u`/`d` up/down a frame, `q` quit. `PYTHONBREAKPOINT=0` disables every `breakpoint()` call.
- **In Jupyter**, after a cell raises, run `%debug` in the next cell to open a post-mortem debugger *at the frame that failed*, with all local variables available. `%pdb on` does that automatically for every error. (Not run here: they wait for keyboard input.)
- **Read tracebacks bottom-up**: the last line is the error; the frame above is where it was raised; walk up to the first frame in *your* code (see [09 · Errors and exceptions](09_errors_and_exceptions.ipynb)).
- **Add context to errors** in data pipelines with `e.add_note(...)` (3.11+), so the traceback says *which record* failed:

In [ ]:
import logging

log = logging.getLogger("nb15.pipeline")
log.handlers.clear()
log.addHandler(logging.StreamHandler(sys.stdout))
log.handlers[0].setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
log.propagate = False
log.setLevel(logging.INFO)

records = ['{"q": "refund?", "label": "billing"}', '{"q": "parcel?", "label": "shipping"}', '{"q": "crash", "label": }']

def load_labels(lines):
    out = []
    for lineno, line in enumerate(lines, start=1):
        log.debug("parsing line %d: %s", lineno, line)             # invisible at INFO
        try:
            out.append(json.loads(line)["label"])
        except json.JSONDecodeError as e:
            e.add_note(f"while parsing line {lineno} of labels.jsonl: {line!r}")
            raise
    return out

try:
    load_labels(records)
except json.JSONDecodeError as e:
    print("error:", e)
    print("note :", e.__notes__[0])
log.setLevel(logging.DEBUG)                                          # turn detail on without editing code
load_labels(records[:1])

## 9. Profiling: measure before you optimise

- `timeit` answers "which of these two versions is faster?" (run many times, take the minimum).
- `cProfile` answers "**where** does the time go?" across a whole call tree; `pstats` sorts and reports it. Look at **tottime** (time inside the function itself) to find the hot spot, **cumtime** (including callees) to find the expensive branch.
- For production: `py-spy` (a sampling profiler you attach to a running process), `tracemalloc` for memory, `line_profiler` for line-by-line timings.

Below, a keyword-scoring function is slower than it should be. Profile first, then fix what the profile points at.

In [ ]:
import cProfile
import pstats
import timeit

import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True})

rng = random.Random(1)
vocab = [f"w{i}" for i in range(2000)]
STOPWORDS_LIST = vocab[:600]                                   # a list: `in` scans it element by element
corpus = [" ".join(rng.choice(vocab) for _ in range(60)) for _ in range(300)]

def tokenize(text):
    return text.lower().split()

def remove_stopwords(tokens, stopwords):
    return [t for t in tokens if t not in stopwords]

def score(doc, query_terms, stopwords):
    return sum(1 for t in remove_stopwords(tokenize(doc), stopwords) if t in query_terms)

def rank(query, stopwords):
    terms = set(tokenize(query))
    return sorted(range(len(corpus)), key=lambda i: score(corpus[i], terms, stopwords), reverse=True)[:5]

profiler = cProfile.Profile()
profiler.enable()
slow_top5 = rank("w1500 w1700 w1999", STOPWORDS_LIST)
profiler.disable()

stats = pstats.Stats(profiler).get_stats_profile()
top = sorted(stats.func_profiles.items(), key=lambda kv: kv[1].tottime, reverse=True)[:5]
print(f"{'function':<45} {'calls':>7} {'tottime s':>10} {'cumtime s':>10}")
for name, fp in top:
    print(f"{name[:45]:<45} {fp.ncalls:>7} {fp.tottime:>10.3f} {fp.cumtime:>10.3f}")

fig, ax = plt.subplots(figsize=(8.5, 3.2))
bars = ax.barh([n[:40] for n, _ in top], [fp.tottime for _, fp in top], color=BLUE, height=0.6)
ax.bar_label(bars, fmt="%.3f s", padding=3, fontsize=9)
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("tottime: seconds spent inside the function itself")
ax.set_title("cProfile: the hot spot is remove_stopwords")
plt.show()
assert top[0][0] == "remove_stopwords"

`remove_stopwords` dominates, and it does almost nothing — except `t not in stopwords`, which on a **list** compares against up to 600 strings for every token. Switch to a `frozenset` (hash lookup, O(1)) and re-measure with `timeit`, checking the result did not change:

In [ ]:
STOPWORDS_SET = frozenset(STOPWORDS_LIST)
fast_top5 = rank("w1500 w1700 w1999", STOPWORDS_SET)
t_slow = min(timeit.repeat(lambda: rank("w1500 w1700 w1999", STOPWORDS_LIST), number=1, repeat=3))
t_fast = min(timeit.repeat(lambda: rank("w1500 w1700 w1999", STOPWORDS_SET), number=1, repeat=3))
print(f"list stopwords: {t_slow * 1000:.0f} ms | set stopwords: {t_fast * 1000:.0f} ms | {t_slow / t_fast:.0f}x faster")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
bars = ax.bar(["before\n(list lookup)", "after\n(set lookup)"], [t_slow * 1000, t_fast * 1000], color=[ORANGE, AQUA], width=0.5)
ax.bar_label(bars, fmt="%.0f ms", padding=2)
ax.set_ylabel("ms to rank 300 docs"); ax.grid(axis="x", visible=False)
ax.set_title("Same ranking, one data-structure change")
plt.show()
assert fast_top5 == slow_top5 and t_fast < t_slow

## Try it yourself

**1.** Write a `unittest.TestCase` for `chunk_text` with `subTest`, covering: an empty string, text shorter than `size`, and invalid `overlap >= size` (use `assertRaises`). Run just that class.

In [ ]:
# Solution — try it yourself first, then run this
class TestChunkText(unittest.TestCase):
    def test_edge_cases(self):
        for text, size, overlap, expected in [("", 5, 1, [""]), ("abc", 5, 1, ["abc"]), ("abcdef", 4, 2, ["abcd", "cdef"])]:
            with self.subTest(text=text, size=size, overlap=overlap):
                self.assertEqual(chunk_text(text, size, overlap), expected)

    def test_bad_overlap(self):
        with self.assertRaises(ValueError):
            chunk_text("abc", size=3, overlap=3)

res = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(TestChunkText))
assert res.wasSuccessful()

**2.** Using a `MagicMock` client, test that `classify_ticket` (a) sends the system prompt first, (b) maps an unexpected reply like `"Refund!!"` to `"other"`, and (c) lets a `TimeoutError` from the client propagate.

In [ ]:
# Solution — try it yourself first, then run this
fake = mock.MagicMock()
fake.chat.completions.create.return_value = completion("Refund!!")
assert classify_ticket("money back please", fake) == "other"
sent = fake.chat.completions.create.call_args.kwargs["messages"]
assert sent[0] == {"role": "system", "content": SYSTEM}

fake.chat.completions.create.side_effect = TimeoutError("read timed out")
try:
    classify_ticket("anything", fake)
    raise AssertionError("expected a TimeoutError")
except TimeoutError:
    print("(a) system prompt first, (b) unknown -> 'other', (c) TimeoutError propagates: all good")

**3.** Write a property test for `parse_label`: for 300 random strings (letters, spaces, punctuation), the result is **always** one of `LABELS`, and a reply that starts with an exact label word always returns that label.

In [ ]:
# Solution — try it yourself first, then run this
rng = random.Random(7)
alphabet = "abcdefghijklmnopqrstuvwxyz .,!?\n"
for _ in range(300):
    junk = "".join(rng.choice(alphabet) for _ in range(rng.randint(0, 30)))
    assert parse_label(junk) in LABELS, junk
    label = rng.choice(LABELS)
    assert parse_label(f"{label.upper()}. {junk}") == label, (label, junk)
print("300 random replies: parse_label always returns a valid label")

## Say it in an interview

- **Testing an LLM feature in CI (pd29):** "Most of the code is not the model. Unit-test prompt building, parsing, validation and retries with a mocked client — free, fast, deterministic. Record real responses once and replay them. Then a small smoke suite on a cheap model per merge with a cost ceiling, and the full eval nightly."
- **Agents (ag28):** "Mock the tools with recorded fixtures, run a fixed task suite, and assert on outcomes and trajectories — task done, no forbidden tool, steps and cost under budget — with the gate expressed as a pass rate over several runs."
- **Reproducibility (in32):** "Pin the model version, temperature 0, seed where supported — and still assert on structure and meaning, not exact strings."
- **Regression (ev22):** "A frozen case set run on every prompt, model or retrieval change; golden files for rendered prompts; property checks for format; track the score over releases to catch slow drift."
- **Mocking:** "`MagicMock` with `return_value`/`side_effect`, `create_autospec` so typos fail, and patch where the name is looked up."
- **Debugging and profiling:** "Reproduce, read the traceback bottom-up, `%debug` post-mortem in notebooks, logging levels instead of prints. Profile with cProfile before optimising; the fix is usually a data structure or an extra round trip, not micro-tuning."
- **Traps:** `assert` for input validation (stripped by `-O`); `assert (x, "msg")` is a tuple and always true; tests that hit the real API; patching the wrong name; flaky timing asserts; a mock without a spec that accepts anything.

## Next

- [16 · Python gotchas interviewers love](16_python_gotchas_interviewers_love.ipynb)
- Evaluation: [eval harness from scratch](../17_interview_coding_rounds/05_eval_harness_from_scratch.ipynb) · [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb) · [A/B testing and statistics for evals](../12_evaluation/05_ab_testing_and_statistics_for_evals.ipynb)
- Production: [observability, logging and tracing](../16_production/08_observability_logging_and_tracing.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [interview bank](../../ai_interview_prep/index.html)